In [27]:
# Cell 1

# Version 14

# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet
/kaggle/input/datasets/shishiradhikari11/casmi-rdkit2025-cp313/rdkit-2025.3.6-cp313-cp313-manylinux_2_28_x86_64.whl
/kaggle/input/datasets/shishiradhikari11/casmi-rdkit2025-cp313/LICENSE-NOTICE.txt
/kaggle/input/datasets/aidensong123/casmi26-coconut-202609/COCONUT_ATTRIBUTION.md
/kaggle/input/datasets/aidensong123/casmi26-coconut-202609/coconut_structures.parquet
/kaggle/input/datasets/salutsirius/enveda-2m4-dataset-from-notebook02a72b3003/checkpoint2m4.pt
/kaggle/input/datasets/salutsirius/enveda-2m4-dataset-from-notebook02a72b3003/.virtual_documents/__notebook_source__.ipynb


In [28]:
# Cell 2

# monoisotopic masses
MASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196,
            P=30.97376163, S=31.97207100, F=18.99840322, Cl=34.96885268,
            Br=78.9183371, I=126.904473, Na=22.9897692809, K=38.96370668,
            Si=27.9769265325, B=11.0093054, Se=79.9165213)
E = 0.00054857990
PROTON = MASS['H'] - E
H2O = 2 * MASS['H'] + MASS['O']
NH4 = MASS['N'] + 4 * MASS['H']
FORMATE = MASS['C'] + 2 * MASS['H'] + 2 * MASS['O']
ACETATE = 2 * MASS['C'] + 4 * MASS['H'] + 2 * MASS['O']

ADDUCTS = {
    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E),
    "[M+Na]+": (1, 1, MASS['Na'] - E), "[M+K]+": (1, 1, MASS['K'] - E),
    "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2 * H2O),
    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),
    "[M+CH3OH+H]+": (1, 1, PROTON + MASS['C'] + 4 * MASS['H'] + MASS['O']),
    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS['C'] + 3 * MASS['H'] + MASS['N']),
    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O),
    "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),
    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON),
    "[M+Cl]-": (1, 1, MASS['Cl'] + E), "[M]-": (1, 1, E),
    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS['Na'] - 2 * PROTON),
    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS['Na'] - E),
    "[2M+NH4]+": (2, 1, NH4 - E), "[2M+K]+": (2, 1, MASS['K'] - E),
    "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),
    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS['Na'] - 2 * PROTON),
    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),
}

def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan)
    ad = np.asarray(adduct, dtype=object)
    for a, (n, z, d) in ADDUCTS.items():
        m = (ad == a)
        if m.any():
            out[m] = (mz[m] * z - d) / n
    return out

In [29]:
# Cell 3

import os, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from pathlib import Path

# !pip install rdkit
!pip install --no-index /kaggle/input/datasets/shishiradhikari11/casmi-rdkit2025-cp313/rdkit-2025.3.6-cp313-cp313-manylinux_2_28_x86_64.whl
# !pip install faiss-cpu
# !pip install --no-index /kaggle/input/datasets/salutsirius/faiss-cpu-notebook9e1116636c/faiss_cpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import rdFingerprintGenerator
import rdkit.Chem as Chem

Processing /kaggle/input/datasets/shishiradhikari11/casmi-rdkit2025-cp313/rdkit-2025.3.6-cp313-cp313-manylinux_2_28_x86_64.whl
rdkit is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [30]:
# Cell 4

class SpectrumTokenizer:
    def __init__(self, mz_min=0.0, mz_max=1800.0, mz_bin_width=0.02, max_peaks=256):
        self.mz_min = mz_min
        self.mz_max = mz_max
        self.mz_bin_width = mz_bin_width
        self.max_peaks = max_peaks
        self.mz_bins = int((mz_max - mz_min) / mz_bin_width) + 1

    def __call__(self, mzs, intensities):
        mzs = np.asarray(mzs)
        intensities = np.asarray(intensities)

        # 1. cap to top-N by intensity
        if len(mzs) > self.max_peaks:
            top = np.argsort(intensities)[-self.max_peaks:]
            mzs = mzs[top]
            intensities = intensities[top]

        # 2. sort kept peaks by m/z
        order = np.argsort(mzs)
        mzs = mzs[order]
        intensities = intensities[order]

        # 3. bin m/z with clipping
        mz_idx = np.clip(
            ((mzs - self.mz_min) / self.mz_bin_width).astype(np.int64),
            0, self.mz_bins - 1
        )

        return {
            'mz_idx': torch.tensor(mz_idx, dtype=torch.long),
            'intensity': torch.tensor(intensities, dtype=torch.float32),
            'length': len(mz_idx),
        }

In [31]:
# Cell 5

class PeakEmbedding(nn.Module):
    def __init__(self, d_model, mz_bins):
        super().__init__()
        self.mz_emb = nn.Embedding(mz_bins, d_model)
        self.intensity_proj = nn.Linear(1, d_model)

    def forward(self, mz_idx, intensity):
        # mz_idx: [B, L], intensity: [B, L]
        mz_vec = self.mz_emb(mz_idx)                       # [B, L, d]
        # intensity.unsqueeze(-1) - [B, L] -> [B, L, 1]
        # (output = input @ weight.T + bias); [B, L, d] = [B, L, 1] @ [1, d]
        int_vec = self.intensity_proj(intensity.unsqueeze(-1))  # [B, L, d] 
        
        return mz_vec + int_vec

In [32]:
# Cell 6

class SpectrumCollator:
    def __init__(self, max_peaks=256, pad_mz_idx=0, pad_intensity=0.0):
        self.max_peaks = max_peaks
        self.pad_mz_idx = pad_mz_idx
        self.pad_intensity = pad_intensity

    def __call__(self, batch):
        # batch: list of dicts from SpectrumTokenizer, each with
        #   'mz_idx': LongTensor [len_i]
        #   'intensity': FloatTensor [len_i]
        #   plus label fields: 'smiles', 'inchikey14', etc.

        B = len(batch)
        L = self.max_peaks

        mz_idx = torch.full((B, L), self.pad_mz_idx, dtype=torch.long)
        intensity = torch.full((B, L), self.pad_intensity, dtype=torch.float32)
        mask = torch.zeros((B, L), dtype=torch.bool)

        for i, sample in enumerate(batch):
            n = min(len(sample['mz_idx']), L)
            mz_idx[i, :n] = sample['mz_idx'][:n]
            intensity[i, :n] = sample['intensity'][:n]
            mask[i, :n] = True

        out = {
            'mz_idx': mz_idx,
            'intensity': intensity,
            'mask': mask,
        }

        # pass through label fields unchanged (as lists)
        for key in batch[0]:
            if key not in ('mz_idx', 'intensity'):
                out[key] = [sample[key] for sample in batch]

        return out

In [33]:
# Cell 7

class MultiHeadAttention(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, causal=False) -> None:
    super().__init__()

    self.d_k = d_k
    self.n_heads = n_heads

    self.key = nn.Linear(d_model, d_k * n_heads)
    self.query = nn.Linear(d_model, d_k * n_heads)
    self.value = nn.Linear(d_model, d_k * n_heads)

    self.fc = nn.Linear(d_k * n_heads, d_model)

    self.causal = causal
    if causal:
      cm = torch.tril(torch.ones(max_len, max_len))
      self.register_buffer(
        'causal_mask',
        cm.view(1, 1, max_len, max_len)
    )


  def forward(self, q, k, v, pad_mask=None):
    q = self.query(q)   # (N, T, d_model) -> (N, T, h*d_k)
    k = self.key(k)   # (N, T, d_model) -> (N, T, h*d_k)
    v = self.value(v)   # (N, T, d_model) -> (N, T, h*d_k)

    N = q.shape[0]
    T_output = q.shape[1]
    T_input = k.shape[1]

    # change the shape
    # (N, T, h, d_k) -> (N, h, T, d_k)
    # in order for matrix multiply to work properly
    q = q.view(N, T_output, self.n_heads, self.d_k).transpose(1, 2)
    k = k.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)
    v = v.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)

    # Compute attention weights
    # (N, h, T, d_k) x (N, h, d_k, T) -> (N, h, T, T)
    attn_scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
    if pad_mask is not None:
      attn_scores = attn_scores.masked_fill(pad_mask[:, None, None, :] == 0, float('-inf'))

    if self.causal:
      attn_scores = attn_scores.masked_fill(self.causal_mask[:, :, :T_output, :T_input] == 0, float('-inf'))

    attn_weights = F.softmax(attn_scores, dim=-1)

    # Compute attention weights values
    # (N, h, T, T) x (N, h, T, d_k) -> (N, h, T, d_k)
    A = attn_weights @ v

    # Reshape it back before final linear layer
    A = A.transpose(1, 2) # (N, T, h, d_k)
    A = A.contiguous().view(N, T_output, self.d_k * self.n_heads) # (N, T, h*d_k)

    # projection
    return self.fc(A)

In [34]:
# Cell 8

class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=256, dropout_prob=0.1) -> None:
    super().__init__()

    self.dropout = nn.Dropout(p=dropout_prob)

    position = torch.arange(max_len).unsqueeze(1)
    exp_term = torch.arange(0, d_model, 2)
    div_term = torch.exp(exp_term * (-math.log(10000.0)/d_model))
    pe = torch.zeros(1, max_len, d_model)
    pe[0, :, 0::2] = torch.sin(position * div_term)
    pe[0, :, 1::2] = torch.cos(position * div_term)
    self.register_buffer('pe', pe)

  def forward(self, x):
    # x.shape (N, T, D)
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)

In [35]:
# Cell 9 

class EncoderBlock(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, dropout_prob=0.1) -> None:
    super().__init__()

    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.mha = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=False)
    self.ann = nn.Sequential(
        nn.Linear(d_model, d_model * 4),
        nn.GELU(),
        nn.Linear(d_model * 4, d_model),
        nn.Dropout(dropout_prob)
    )
    self.dropout = nn.Dropout(p=dropout_prob)

  def forward(self, x, mask=None):
    x = self.ln1(x + self.mha(x, x, x, mask))
    x = self.ln2(x + self.ann(x))
    x = self.dropout(x)
    return x

In [36]:
# Cell 10

class Encoder(nn.Module):
  def __init__(self, mz_bins, max_len, d_k, d_model, n_heads, n_layers,
               #n_classes,
               dropout_prob) -> None:
    super().__init__()

    # self.embedding = nn.Embedding(vocab_size, d_model)
    self.embedding = PeakEmbedding(d_model, mz_bins)
    self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)
    transformer_blocks = [
        EncoderBlock(d_k, d_model, n_heads, max_len, dropout_prob) for _ in range(n_layers)]
    self.transformer_blocks = nn.Sequential(*transformer_blocks)
    self.ln = nn.LayerNorm(d_model)
    #self.fc = nn.Linear(d_model, n_classes)

  def forward(self, mz_idx, intensity, mask=None):
    x = self.embedding(mz_idx, intensity)
    x = self.pos_encoding(x)

    for block in self.transformer_blocks:
      x = block(x, mask)

    # many-to-one x has shape (N x T x D) -> (N x D)
    #x = x[:, 0, :]

    # Masked mean pooling — averages the encoder output over real peaks only, ignoring padding
    mask_expanded = mask.unsqueeze(-1).float()  # [B, L, 1]
    x = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1)
      
    x = self.ln(x)
    #x = self.fc(x)
    return x

In [37]:
# Cell 11

from rdkit.Chem import rdFingerprintGenerator
import rdkit.Chem as Chem

class MorganEncoder:
    def __init__(self, radius=2, n_bits=2048):
        self.gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

    def __call__(self, smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        fp = self.gen.GetFingerprintAsNumPy(mol)   # returns np.uint8 array directly
        return fp.astype(np.float32)

In [38]:
# Cell 12

class ProjectionHead(nn.Module):
    def __init__(self, d_model=256, d_fp=2048):
        super().__init__()
        self.proj = nn.Linear(d_model, d_fp)

    def forward(self, x):
        return self.proj(x)

In [39]:
# Cell 13

from torch.utils.data import Dataset

class SpectrumDataset(Dataset):
    def __init__(self, df, tokenizer, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        sample = self.tokenizer(
            row['ms2_mzs'],
            row['ms2_normalized_intensities'],
        )
        if self.is_test:
            sample['molecule_id'] = row['molecule_id']
        else:
            sample['smiles'] = row['normalized_smiles']
            sample['inchikey14'] = row['inchikey14']
        return sample



In [40]:
# Cell 14

# load train (minimal: pool needs only ik + smiles), load test
train = pd.read_parquet(
    '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet',
    columns=['inchikey14', 'normalized_smiles']
)

test = pd.read_parquet(
    '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet',
    columns=['molecule_id', 'ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz', 'adduct']
)

In [41]:
# Cell 15

# instantiate and load checkpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = SpectrumTokenizer()
collate = SpectrumCollator(max_peaks=256)
me = MorganEncoder()

enc = Encoder(
    mz_bins=tok.mz_bins,
    max_len=256,
    d_k=64,
    d_model=256,
    n_heads=4,
    n_layers=4,
    dropout_prob=0.1,
).to(device)
proj = ProjectionHead().to(device)

matches = list(Path("/kaggle/input").rglob("checkpoint2m4.pt"))
print("checkpoints found:", matches)
state = torch.load(matches[0], map_location=device)
enc.load_state_dict(state['enc'])
proj.load_state_dict(state['proj'])
enc.eval()
proj.eval()

checkpoints found: [PosixPath('/kaggle/input/datasets/salutsirius/enveda-2m4-dataset-from-notebook02a72b3003/checkpoint2m4.pt')]


ProjectionHead(
  (proj): Linear(in_features=256, out_features=2048, bias=True)
)

In [42]:
# Cell 16

# pool

import gc
from rdkit.Chem.Descriptors import ExactMolWt

# --- training molecules ---
train_mols = train.drop_duplicates('inchikey14')[['inchikey14', 'normalized_smiles']].reset_index(drop=True)
print(f'train molecules: {len(train_mols)}')
train_ik_list = train_mols['inchikey14'].tolist()
train_smiles_list = train_mols['normalized_smiles'].tolist()

# --- COCONUT ---
coconut = pd.read_parquet(
    '/kaggle/input/datasets/aidensong123/casmi26-coconut-202609/coconut_structures.parquet',
    columns=['canonical_smiles', 'inchikey']
).dropna(subset=['canonical_smiles', 'inchikey']).reset_index(drop=True)
print(f'coconut rows: {len(coconut)}')
coconut_ik_list = coconut['inchikey'].str[:14].tolist()
coconut_smiles_list = coconut['canonical_smiles'].tolist()

# --- combined pool (lists only) ---
all_ik_list = train_ik_list + coconut_ik_list
all_smiles_list = train_smiles_list + coconut_smiles_list
print(f'combined pool: {len(all_smiles_list)} molecules')

del coconut, train_mols
gc.collect()

# --- exact neutral mass per pool molecule ---
pool_mass = np.array([
    ExactMolWt(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else np.nan
    for s in tqdm(all_smiles_list, desc='pool masses')
])
print(f'mass finite: {np.isfinite(pool_mass).mean():.3f}')

# --- sort by mass ---
order = np.argsort(np.where(np.isfinite(pool_mass), pool_mass, 1e18), kind='mergesort')
pool_mass_sorted = pool_mass[order]
pool_smiles_sorted = [all_smiles_list[i] for i in order]
pool_ik_sorted = [all_ik_list[i] for i in order]
ik_to_smiles = dict(zip(all_ik_list, all_smiles_list))

# free the unsorted copies
del pool_mass, all_ik_list
gc.collect()

def mass_window(target, ppm=10.0):
    lo = np.searchsorted(pool_mass_sorted, target * (1 - ppm / 1e6), 'left')
    hi = np.searchsorted(pool_mass_sorted, target * (1 + ppm / 1e6), 'right')
    return lo, hi

print('done')

train molecules: 275810
coconut rows: 738827
combined pool: 1014637 molecules


pool masses:   0%|          | 0/1014637 [00:00<?, ?it/s]

mass finite: 1.000
done


In [43]:
# Cell 17

# ============================================================
# Channel 1 setup: spectral library + Numba entropy-similarity kernels
# Built ONCE from full train.parquet. Nothing is withheld.
# ============================================================
import pyarrow as pa
import pyarrow.parquet as pq
from numba import njit, prange

MZ_TOL       = 0.01
C1_INT_FLOOR = 0.002
C1_MAX_PEAKS = 256
C1_INT_POWER = 1.0
C1_ENT_W     = True
TAU          = 0.9999   # Channel 1 override threshold

@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n = len(mz)
    if n == 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    mx = 0.0
    for i in range(n):
        if it[i] > mx: mx = it[i]
    if mx <= 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    thr = floor * mx
    c = 0
    for i in range(n):
        if it[i] >= thr: c += 1
    idx = np.empty(c, np.int64); j = 0
    for i in range(n):
        if it[i] >= thr:
            idx[j] = i; j += 1
    if c > topk:
        v = np.empty(c, np.float32)
        for i in range(c): v[i] = it[idx[i]]
        o = np.argsort(v)[c - topk:]
        k2 = np.empty(topk, np.int64)
        for i in range(topk): k2[i] = idx[o[i]]
        k2.sort(); idx = k2; c = topk
    om = np.empty(c, np.float32); oi = np.empty(c, np.float32); s = 0.0
    for i in range(c):
        om[i] = mz[idx[i]]
        v = it[idx[i]] ** power
        oi[i] = v; s += v
    if s > 0:
        for i in range(c): oi[i] /= s
    if ent_weight:
        S = 0.0
        for i in range(c):
            if oi[i] > 0: S -= oi[i] * np.log(oi[i])
        if S < 3.0:
            w = 0.25 + 0.25 * S
            s2 = 0.0
            for i in range(c):
                oi[i] = oi[i] ** w; s2 += oi[i]
            if s2 > 0:
                for i in range(c): oi[i] /= s2
    order = np.argsort(om)
    om = om[order]
    oi = oi[order]
    return om, oi

@njit(cache=True, fastmath=True)
def _ent(qmz, qp, cmz, cp, tol):
    i = 0; j = 0; n = len(qmz); m = len(cmz)
    SA = 0.0
    for x in range(n):
        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])
    SB = 0.0
    for x in range(m):
        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])
    SAB = 0.0; tot = 0.0
    buf = np.empty(n + m, np.float64); b = 0
    while i < n and j < m:
        dd = qmz[i] - cmz[j]
        if dd < -tol:
            buf[b] = qp[i]; i += 1; b += 1
        elif dd > tol:
            buf[b] = cp[j]; j += 1; b += 1
        else:
            buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1
    while i < n: buf[b] = qp[i]; i += 1; b += 1
    while j < m: buf[b] = cp[j]; j += 1; b += 1
    for x in range(b): tot += buf[x]
    if tot <= 0: return 0.0
    for x in range(b):
        v = buf[x] / tot
        if v > 0: SAB -= v * np.log(v)
    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)

@njit(cache=True, fastmath=True)
def _ent_shift(qmz, qp, cmz, cp, tol, shift):
    a = _ent(qmz, qp, cmz, cp, tol)
    if abs(shift) < 1e-3:
        return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)):
        sm[i] = cmz[i] + shift
    b = _ent(qmz, qp, sm, cp, tol)
    return a if a > b else b

@njit(cache=True, fastmath=True, parallel=True)
def _search_shift(qmz, qp, cand, off, allmz, allin, tol, shift):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]
        a, b = off[c], off[c+1]
        if b <= a: continue
        out[k] = _ent_shift(qmz, qp, allmz[a:b], allin[a:b], tol, shift[k])
    return out

# ---- build library from FULL train.parquet (no withholding) ----
TRAIN_PATH = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet'
_tab = pq.read_table(
    TRAIN_PATH,
    columns=['inchikey14', 'adduct', 'precursor_mz',
             'ms2_mzs', 'ms2_normalized_intensities'],
)

_lib_prec = _tab.column('precursor_mz').to_numpy(zero_copy_only=False).astype(np.float64)
_lib_add  = np.asarray(_tab.column('adduct').cast(pa.string()).to_pylist(), dtype=object)
lib_ik    = np.asarray(_tab.column('inchikey14').cast(pa.string()).to_pylist(), dtype=str)

_lib_nm = neutral_mass(_lib_prec, _lib_add)

_ok = np.isfinite(_lib_nm) & (_lib_nm > 50.0) & (_lib_nm < 2000.0)
lib_order     = np.argsort(np.where(_ok, _lib_nm, 1e18), kind='mergesort')
lib_nm_sorted = _lib_nm[lib_order]
lib_n_ok      = int(_ok.sum())

_mzc = _tab.column('ms2_mzs').combine_chunks()
_itc = _tab.column('ms2_normalized_intensities').combine_chunks()
lib_off = _mzc.offsets.to_numpy().astype(np.int64)
lib_mz  = _mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)
lib_in  = _itc.values.to_numpy(zero_copy_only=False).astype(np.float32)

del _tab, _mzc, _itc
import gc; gc.collect()

print(f'[Channel 1] library spectra: {len(lib_off)-1:,}')
print(f'[Channel 1] library structures: {len(np.unique(lib_ik)):,}')
print(f'[Channel 1] finite-mass spectra: {lib_n_ok:,}')
print(f'[Channel 1] TAU = {TAU}')

def channel1_best_ik(target, spec_iter, ppm_win=10.0, ppm_fb=35.0):
    """Return (best_ik, best_sim) for this molecule; (None, 0.0) if nothing found."""
    lo = np.searchsorted(lib_nm_sorted[:lib_n_ok], target * (1 - ppm_win/1e6), 'left')
    hi = np.searchsorted(lib_nm_sorted[:lib_n_ok], target * (1 + ppm_win/1e6), 'right')
    cand = lib_order[lo:hi]
    if len(cand) == 0:
        lo = np.searchsorted(lib_nm_sorted[:lib_n_ok], target * (1 - ppm_fb/1e6), 'left')
        hi = np.searchsorted(lib_nm_sorted[:lib_n_ok], target * (1 + ppm_fb/1e6), 'right')
        cand = lib_order[lo:hi]
    if len(cand) == 0:
        return None, 0.0

    cand_ik = lib_ik[cand]
    hits = {}
    for mz, it in spec_iter:
        qm, qp = _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                        C1_INT_FLOOR, C1_MAX_PEAKS, C1_INT_POWER, C1_ENT_W)
        if len(qm) == 0:
            continue
        for k in range(len(cand)):
            c = cand[k]
            a = lib_off[c]; b = lib_off[c+1]
            if b <= a: continue
            cm, cp = _clean(lib_mz[a:b], lib_in[a:b],
                            C1_INT_FLOOR, C1_MAX_PEAKS, C1_INT_POWER, C1_ENT_W)
            if len(cm) == 0: continue
            s = _ent(qm, qp, cm, cp, MZ_TOL)
            key = str(cand_ik[k])
            if s > hits.get(key, -1.0):
                hits[key] = float(s)

    if not hits:
        return None, 0.0
    best_ik = max(hits.items(), key=lambda x: x[1])
    return best_ik[0], best_ik[1]

[Channel 1] library spectra: 2,539,608
[Channel 1] library structures: 275,810
[Channel 1] finite-mass spectra: 2,537,242
[Channel 1] TAU = 0.9999


In [44]:
# Cell 17a — representative library + analog propagation scoring
# Builds one richest spectrum per train inchikey (mass-sorted), then scores
# COCONUT candidates by Tanimoto-weighted propagation from shifted-spectrum analogs.

import gc

# ---- 1. representative library: one richest spectrum per inchikey ----
_npk = np.diff(lib_off)
_best = {}
for _i in range(len(lib_ik)):
    _k = lib_ik[_i]
    if _k not in _best or _npk[_i] > _npk[_best[_k]]:
        _best[_k] = _i
_rep_idx = np.array(sorted(_best.values()))
del _best

_pos_in_order = np.full(len(lib_ik), -1, dtype=np.int64)
_pos_in_order[lib_order[:lib_n_ok]] = np.arange(lib_n_ok)

_rep_rows = _rep_idx[np.isfinite(lib_nm_sorted[:lib_n_ok][_pos_in_order[_rep_idx]])]
_rep_rows = _rep_rows[np.argsort(lib_nm_sorted[:lib_n_ok][_pos_in_order[_rep_rows]], kind='mergesort')]
rep_ik = lib_ik[_rep_rows]
rep_nm = lib_nm_sorted[:lib_n_ok][_pos_in_order[_rep_rows]]

# clean each representative spectrum once (sort + entropy-weight)
_rep_mz_list, _rep_in_list, _rep_lens = [], [], []
for i in _rep_rows:
    a, b = lib_off[i], lib_off[i+1]
    cm, cp = _clean(lib_mz[a:b], lib_in[a:b],
                    C1_INT_FLOOR, C1_MAX_PEAKS, C1_INT_POWER, C1_ENT_W)
    _rep_mz_list.append(cm)
    _rep_in_list.append(cp)
    _rep_lens.append(len(cm))

rep_mz = np.concatenate(_rep_mz_list).astype(np.float32) if _rep_mz_list else np.empty(0, np.float32)
rep_in = np.concatenate(_rep_in_list).astype(np.float32) if _rep_in_list else np.empty(0, np.float32)
rep_off = np.zeros(len(_rep_lens) + 1, dtype=np.int64)
rep_off[1:] = np.cumsum(_rep_lens)
del _rep_mz_list, _rep_in_list, _rep_lens

assert np.all(np.diff(rep_nm) >= 0), 'rep_nm not sorted'
print(f'representative library: {len(rep_ik):,} structures, mass {rep_nm[0]:.1f}..{rep_nm[-1]:.1f}')

del _npk, _rep_idx, _rep_rows, _pos_in_order
gc.collect()

# ---- 2. analog scores ----
ANALOG_WIN = 200.0   # ±Da
N_ANALOG   = 80      # top analogs kept
SIM_POWER  = 4.0     # weight = sim^p

def analog_scores(target, spec_iter, topn=25):
    """
    target: neutral mass
    spec_iter: list of (mz_array, intensity_array)
    returns: list of (inchikey14, analog_score) for COCONUT candidates in the ±10 ppm window,
             ranked by analog_score. Empty list if nothing found.
    """
    lo = np.searchsorted(rep_nm, target - ANALOG_WIN, 'left')
    hi = np.searchsorted(rep_nm, target + ANALOG_WIN, 'right')
    if hi <= lo:
        return []
    cand_rep = np.arange(lo, hi, dtype=np.int64)
    shift = (target - rep_nm[lo:hi]).astype(np.float32)

    best_sim = np.full(len(cand_rep), -1.0, dtype=np.float32)
    for mz, it in spec_iter:
        qm, qp = _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                        C1_INT_FLOOR, C1_MAX_PEAKS, C1_INT_POWER, C1_ENT_W)
        if len(qm) == 0:
            continue
        s = _search_shift(qm, qp, cand_rep, rep_off, rep_mz, rep_in, MZ_TOL, shift)
        np.maximum(best_sim, s, out=best_sim)

    order = np.argsort(-best_sim)[:N_ANALOG]
    analog_sims = best_sim[order]
    keep = analog_sims > 0.05
    if not keep.any():
        return []
    analog_iks = rep_ik[lo + order][keep]
    analog_sims = analog_sims[keep]

    lo_c, hi_c = mass_window(target, ppm=10.0)
    if hi_c <= lo_c:
        lo_c, hi_c = mass_window(target, ppm=35.0)
    if hi_c <= lo_c:
        return []
    cand_iks = pool_ik_sorted[lo_c:hi_c]
    cand_smiles = pool_smiles_sorted[lo_c:hi_c]

    _seen = set()
    _uniq_ik, _uniq_smi = [], []
    for ik, smi in zip(cand_iks, cand_smiles):
        if ik not in _seen:
            _seen.add(ik)
            _uniq_ik.append(ik)
            _uniq_smi.append(smi)
    cand_iks = _uniq_ik
    cand_smiles = _uniq_smi
    if not cand_iks:
        return []

    cfps = []
    for s in cand_smiles:
        fp = me(s)
        cfps.append(fp if fp is not None else np.zeros(2048, np.float32))
    cfps = np.stack(cfps).astype(np.float32)
    cfps /= (np.linalg.norm(cfps, axis=1, keepdims=True) + 1e-9)

    afps = []
    for ik in analog_iks:
        smi = ik_to_smiles.get(ik)
        fp = me(smi) if smi is not None else np.zeros(2048, np.float32)
        afps.append(fp)
    afps = np.stack(afps).astype(np.float32)
    afps /= (np.linalg.norm(afps, axis=1, keepdims=True) + 1e-9)

    tan = cfps @ afps.T
    weights = (analog_sims.astype(np.float32) ** SIM_POWER)
    scores = tan @ weights

    idx = np.argsort(-scores)[:topn]
    return [(cand_iks[i], float(scores[i])) for i in idx if scores[i] > 0]

# ---- 3. smoke test on 3 molecules ----
print('\n=== analog smoke test (3 molecules) ===')
_te = test.copy()
_te['nm'] = neutral_mass(_te.precursor_mz.values.astype(np.float64), _te.adduct.values)
for _mid, _sub in list(_te.groupby('molecule_id', sort=False))[:3]:
    _nms = _sub.nm.values[np.isfinite(_sub.nm.values)]
    if len(_nms) == 0:
        print(f'{_mid}: no mass'); continue
    _target = float(np.median(_nms))
    _spec = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in _sub.itertuples()]
    _res = analog_scores(_target, _spec, topn=5)
    print(f'\n{_mid}  target={_target:.3f}  n_hits={len(_res)}')
    for _ik, _s in _res:
        print(f'   {_ik:14s}  analog={_s:.4f}')

representative library: 275,810 structures, mass 58.0..1999.9

=== analog smoke test (3 molecules) ===

m_005e53  target=374.064  n_hits=5
   QCTQNXJUUXYSRK  analog=3.4720
   SCIHCHFEKPRVSD  analog=2.9712
   YYPOJIOVHJAXPU  analog=2.8042
   XZGCKUAAHQVVHY  analog=2.7972
   HATBONOIGREPOF  analog=2.7200

m_006153  target=341.149  n_hits=5
   BKIHGKNEJKYYAC  analog=3.9469
   OWLWDUZRCJYSHF  analog=3.7799
   SSOWPHRKXKLBRF  analog=3.3782
   BJVQDFXKIABGOF  analog=3.3263
   WULWOWQOXGVOOC  analog=3.3096

m_00b5aa  target=306.195  n_hits=5
   RWCPIENWOXGKAV  analog=2.4408
   JONXRUGQRMUWSW  analog=1.7648
   HFMCJNNFYNLPGM  analog=1.7369
   YZLBMOSJNRZHFZ  analog=1.7230
   VERRYRPCZDWOJY  analog=1.7173


In [45]:
# Cell 18

from collections import defaultdict

test['neutral_mass'] = neutral_mass(
    test['precursor_mz'].values.astype(np.float64),
    test['adduct'].values
)

mols = list(test.groupby('molecule_id', sort=False))

transformer_scores = {}   # mid -> {ik: rrf_score}
n_fallback = 0
n_c1_none = 0
all_c1_sims = []

for gi, (mid, sub) in enumerate(tqdm(mols, desc='transformer')):
    nms = sub['neutral_mass'].values[np.isfinite(sub['neutral_mass'].values)]
    if len(nms) == 0:
        transformer_scores[mid] = {}
        continue
    target = float(np.median(nms))

    spec_iter = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]
    c1_ik, c1_sim = channel1_best_ik(target, spec_iter)
    if c1_ik is None:
        n_c1_none += 1
    else:
        all_c1_sims.append(float(c1_sim))

    lo, hi = mass_window(target, ppm=10.0)
    if hi <= lo:
        lo, hi = mass_window(target, ppm=35.0)
        n_fallback += 1
    if hi <= lo:
        transformer_scores[mid] = {}
        continue

    cand_iks = pool_ik_sorted[lo:hi]
    cand_fps = np.stack([
        me(s) if me(s) is not None else np.zeros(2048, np.float32)
        for s in pool_smiles_sorted[lo:hi]
    ]).astype('float32')
    cand_fps /= np.linalg.norm(cand_fps, axis=1, keepdims=True) + 1e-9

    spectrum_scores = defaultdict(float)
    for r in sub.itertuples():
        s = tok(r.ms2_mzs, r.ms2_normalized_intensities)
        mz_t = s['mz_idx'].unsqueeze(0).to(device)
        int_t = s['intensity'].unsqueeze(0).to(device)
        mask_t = torch.ones_like(mz_t, dtype=torch.bool, device=device)

        with torch.no_grad():
            q = proj(enc(mz_t, int_t, mask_t))
            q = F.normalize(q, dim=-1).cpu().numpy().astype('float32')

        sims = (cand_fps @ q.T).squeeze(-1)
        order_sims = np.argsort(-sims)
        for rank, idx in enumerate(order_sims, start=1):
            spectrum_scores[cand_iks[idx]] += 1.0 / rank

    transformer_scores[mid] = dict(spectrum_scores)

print(f'fallback used: {n_fallback}/{len(mols)}')
print(f'Channel 1 found no candidates: {n_c1_none}/{len(mols)}')
print(f'transformer scores stored for {len(transformer_scores)} molecules')

transformer:   0%|          | 0/400 [00:00<?, ?it/s]

fallback used: 0/400
Channel 1 found no candidates: 0/400
transformer scores stored for 400 molecules


In [46]:
# Cell 18a — merge transformer + analog, produce submission

from collections import defaultdict

# merge weights — tune these if the first submission is ambiguous
W_TRANSFORMER = 0.7
W_ANALOG      = 0.3

submission_rows = []
n_analog_empty = 0
n_transformer_empty = 0
n_both_empty = 0
n_c1_promoted = 0
n_merged = 0

for gi, (mid, sub) in enumerate(tqdm(mols, desc='merge')):
    nms = sub['neutral_mass'].values[np.isfinite(sub['neutral_mass'].values)]

    # Channel 1 for the exact-match override
    if len(nms) == 0:
        submission_rows.append({'molecule_id': mid, 'smiles': 'CCO'})
        continue
    target = float(np.median(nms))
    spec_iter = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]
    c1_ik, c1_sim = channel1_best_ik(target, spec_iter)

    # transformer scores
    t_scores = transformer_scores.get(mid, {})

    # analog scores
    a_hits = analog_scores(target, spec_iter, topn=100)
    a_scores = dict(a_hits) if a_hits else {}

    # normalize each channel independently
    def _norm(d):
        if not d:
            return {}
        v = np.array(list(d.values()), dtype=np.float32)
        mn, mx = v.min(), v.max()
        if mx - mn < 1e-9:
            return {k: 1.0 for k in d}
        return {k: (val - mn) / (mx - mn) for k, val in d.items()}

    tn = _norm(t_scores)
    an = _norm(a_scores)

    if not tn and not an:
        n_both_empty += 1
        submission_rows.append({'molecule_id': mid, 'smiles': 'CCO'})
        continue
    if not tn:
        n_transformer_empty += 1
    if not an:
        n_analog_empty += 1

    # combined score: union of keys
    keys = set(tn) | set(an)
    combined = {k: W_TRANSFORMER * tn.get(k, 0.0) + W_ANALOG * an.get(k, 0.0) for k in keys}

    top25 = sorted(combined.items(), key=lambda x: -x[1])[:25]
    smiles_25 = [ik_to_smiles.get(ik, 'CCO') for ik, _ in top25]
    n_merged += 1

    # Channel 1 exact-match override
    if c1_ik is not None and c1_sim > TAU:
        c1_smiles = ik_to_smiles.get(c1_ik)
        if c1_smiles is not None:
            smiles_25 = [c1_smiles] + [s for s in smiles_25 if s != c1_smiles]
            smiles_25 = smiles_25[:25]
            n_c1_promoted += 1

    while len(smiles_25) < 25:
        smiles_25.append(smiles_25[-1] if smiles_25 else 'CCO')

    submission_rows.append({'molecule_id': mid, 'smiles': ';'.join(smiles_25)})

sub = pd.DataFrame(submission_rows)
print(f'\nmerged: {n_merged}/{len(mols)}')
print(f'analog empty (transformer only): {n_analog_empty}/{len(mols)}')
print(f'transformer empty (analog only): {n_transformer_empty}/{len(mols)}')
print(f'both empty (CCO): {n_both_empty}/{len(mols)}')
print(f'Channel 1 promoted to rank 1: {n_c1_promoted}/{len(mols)}')
print(sub.shape)
print(sub.head())

assert set(sub['molecule_id']) == set(test['molecule_id']), "missing or extra molecule_id"
max_guesses = sub['smiles'].apply(lambda s: len(s.split(';'))).max()
assert max_guesses <= 25, f"too many guesses: {max_guesses}"
sub.to_csv('submission_merged.csv', index=False)
print('written submission_merged.csv')

merge:   0%|          | 0/400 [00:00<?, ?it/s]


merged: 400/400
analog empty (transformer only): 0/400
transformer empty (analog only): 0/400
both empty (CCO): 0/400
Channel 1 promoted to rank 1: 400/400
(400, 2)
  molecule_id                                             smiles
0    m_005e53  CN(Cc1ncccc1C(=O)O)CC1(c2ccc(Br)cc2)CC1;O=C(O)...
1    m_006153  COc1ccc(N2CC(C(=O)Nc3nncn3C3CC3)CC2=O)cc1;CCn1...
2    m_00b5aa  COc1ccnc(CN2CCOCC3(CCCC3)C2)c1OC;Cc1ccccc1OCCC...
3    m_01e922  Cc1ccc(C(=O)OCC2CCCCC2)c(=O)[nH]1;Cc1cc(O)ccc1...
4    m_0259d4  O=C(NCCCn1ccnc1)NCCc1coc(-c2ccccc2)n1;Cc1cnn(C...
written submission_merged.csv


In [47]:
# Cell 19

print(test['adduct'].value_counts())
print([a for a in test['adduct'].unique() if a not in ADDUCTS])

adduct
[M+H]+          959
[M-H]-          193
[M+CH2O2-H]-     31
[M+Na]+          22
[M+NH4]+          4
[M+K]+            2
[M+Cl]-           2
Name: count, dtype: int64
[]


In [48]:
# Cell 19a — feature extraction on Pool B (holdout) for LightGBM ranker
# One row per (holdout molecule, candidate in its mass window), labeled by true ik.

import gc
import numpy as np
import pandas as pd
from collections import defaultdict
from rdkit.Chem.Descriptors import ExactMolWt
from rdkit import Chem
from tqdm.auto import tqdm

# ---- 1. rebuild the 2,000-molecule holdout (same split as validation) ----
train_full = pd.read_parquet(
    '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet',
    columns=['inchikey14', 'normalized_smiles', 'ms2_mzs',
             'ms2_normalized_intensities', 'precursor_mz', 'adduct']
)
unique_mols = train_full['inchikey14'].unique()
rng = np.random.default_rng(42)
val_mols_array = rng.choice(unique_mols, size=int(0.05 * len(unique_mols)), replace=False)
val_mols_set = set(val_mols_array)
val_df = train_full[train_full['inchikey14'].isin(val_mols_set)].reset_index(drop=True)

VAL_COUNTS = {1:275, 2:485, 3:495, 4:530, 5:120, 6:70, 7:10, 8:10, 9:5}
val_mol_counts = val_df.groupby('inchikey14').size()
rng = np.random.default_rng(42)
selected = []
for n, k in VAL_COUNTS.items():
    eligible = np.sort(val_mol_counts[val_mol_counts == n].index.to_numpy())
    selected.extend(rng.choice(eligible, size=k, replace=False).tolist())
HOLDOUT_IKS = [str(x) for x in selected]
print(f'holdout molecules: {len(HOLDOUT_IKS):,}')

val_df['neutral_mass'] = neutral_mass(
    val_df['precursor_mz'].values.astype(np.float64),
    val_df['adduct'].values
)
val_df = val_df[val_df['inchikey14'].isin(HOLDOUT_IKS)].reset_index(drop=True)

# ---- 2. feature extraction per molecule ----
rows = []
n_skipped_mass = 0
n_skipped_window = 0

mols = list(val_df.groupby('inchikey14', sort=False))
for gi, (mid, sub) in enumerate(tqdm(mols, desc='pool B')):
    nms = sub['neutral_mass'].values[np.isfinite(sub['neutral_mass'].values)]
    if len(nms) == 0:
        n_skipped_mass += 1
        continue
    target = float(np.median(nms))

    # --- candidate window ---
    lo, hi = mass_window(target, ppm=10.0)
    if hi <= lo:
        lo, hi = mass_window(target, ppm=35.0)
    if hi <= lo:
        n_skipped_window += 1
        continue
    cand_iks = pool_ik_sorted[lo:hi]
    cand_smiles = pool_smiles_sorted[lo:hi]

    # dedup (ik, smiles)
    _seen = set()
    u_ik, u_smi = [], []
    for ik, smi in zip(cand_iks, cand_smiles):
        if ik not in _seen:
            _seen.add(ik)
            u_ik.append(ik)
            u_smi.append(smi)
    cand_iks, cand_smiles = u_ik, u_smi
    if not cand_iks:
        n_skipped_window += 1
        continue

    # --- transformer scores ---
    spec_iter = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]
    cand_fps = np.stack([
        me(s) if me(s) is not None else np.zeros(2048, np.float32)
        for s in cand_smiles
    ]).astype('float32')
    cand_fps /= (np.linalg.norm(cand_fps, axis=1, keepdims=True) + 1e-9)

    spectrum_scores = defaultdict(float)
    for r in sub.itertuples():
        s = tok(r.ms2_mzs, r.ms2_normalized_intensities)
        mz_t = s['mz_idx'].unsqueeze(0).to(device)
        int_t = s['intensity'].unsqueeze(0).to(device)
        mask_t = torch.ones_like(mz_t, dtype=torch.bool, device=device)
        with torch.no_grad():
            q = proj(enc(mz_t, int_t, mask_t))
            q = F.normalize(q, dim=-1).cpu().numpy().astype('float32')
        sims = (cand_fps @ q.T).squeeze(-1)
        order_sims = np.argsort(-sims)
        for rank, idx in enumerate(order_sims, start=1):
            spectrum_scores[cand_iks[idx]] += 1.0 / rank

    t_max = max(spectrum_scores.values()) if spectrum_scores else 1.0

    # --- analog scores ---
    a_hits = analog_scores(target, spec_iter, topn=len(cand_iks))
    a_map = dict(a_hits) if a_hits else {}
    a_max = max(a_map.values()) if a_map else 1.0

    # --- Channel 1 ---
    c1_ik, c1_sim = channel1_best_ik(target, spec_iter)

    # --- one row per candidate ---
    for ik, smi in zip(cand_iks, cand_smiles):
        ppm_err = abs(ExactMolWt(Chem.MolFromSmiles(smi)) - target) / target * 1e6 \
                  if Chem.MolFromSmiles(smi) is not None else 1e6
        rows.append({
            'molecule_id': mid,
            'inchikey14': ik,
            'label': 1 if ik == mid else 0,
            't_score': float(spectrum_scores.get(ik, 0.0)),
            't_norm': float(spectrum_scores.get(ik, 0.0)) / (t_max + 1e-9),
            'a_score': float(a_map.get(ik, 0.0)),
            'a_norm': float(a_map.get(ik, 0.0)) / (a_max + 1e-9),
            'mass_ppm': float(ppm_err),
            'n_cand': len(cand_iks),
            'c1_is_true': 1 if (c1_ik is not None and c1_ik == mid) else 0,
            'c1_sim': float(c1_sim) if c1_ik is not None else 0.0,
            'target_mass': target,
        })

feat_df = pd.DataFrame(rows)
print(f'\nrows: {len(feat_df):,}')
print(f'molecules with rows: {feat_df.molecule_id.nunique():,}')
print(f'skipped (no mass): {n_skipped_mass}, (no window): {n_skipped_window}')
print(f'positive rate: {feat_df.label.mean():.4f}')
print(feat_df.head())

feat_df.to_parquet('/kaggle/working/pool_b_features.parquet', index=False)
print('written pool_b_features.parquet')

holdout molecules: 2,000


pool B:   0%|          | 0/2000 [00:00<?, ?it/s]


rows: 213,709
molecules with rows: 1,990
skipped (no mass): 4, (no window): 6
positive rate: 0.0091
      molecule_id      inchikey14  label   t_score    t_norm   a_score  \
0  KFAKESMKRPNZTM  MQIMZDXIAHJKQP      0  1.000000  1.000000  0.227825   
1  KFAKESMKRPNZTM  IFIDBGOEIPJQOL      0  0.166667  0.166667  0.214797   
2  KFAKESMKRPNZTM  MGGUBEAYSYYHRV      0  0.125000  0.125000  0.198068   
3  KFAKESMKRPNZTM  ODZZNDLBTYRTEG      0  0.142857  0.142857  0.263953   
4  KFAKESMKRPNZTM  GXJMTRHFAWEIDK      0  0.200000  0.200000  0.619081   

     a_norm  mass_ppm  n_cand  c1_is_true  c1_sim  target_mass  
0  0.193096  8.145225      17           1     1.0   271.066679  
1  0.182054  3.929022      17           1     1.0   271.066679  
2  0.167876  3.929022      17           1     1.0   271.066679  
3  0.223717  3.929022      17           1     1.0   271.066679  
4  0.524710  3.929022      17           1     1.0   271.066679  
written pool_b_features.parquet


In [60]:
# Cell 19b — train LightGBM ranker on Pool B features (v2: no label leak)
# FIX: dropped 'c1_is_true' (uncomputable at inference — needs true inchikey).
#      'c1_sim' now read as per-molecule constant (matches 19a/19c fix).
# Drops molecules with 0 positives, splits by molecule, trains HistGradientBoosting,
# evaluates MRR@25 on held-out molecules.

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
import pickle

feat_df = pd.read_parquet('/kaggle/working/pool_b_features.parquet')
print(f'raw rows: {len(feat_df):,}  molecules: {feat_df.molecule_id.nunique():,}')

# ---- drop molecules with 0 positives ----
pos_per_mol = feat_df.groupby('molecule_id')['label'].transform('sum')
feat_df = feat_df[pos_per_mol > 0].reset_index(drop=True)
print(f'after filter: rows {len(feat_df):,}  molecules {feat_df.molecule_id.nunique():,}')

# ---- split by molecule (no leakage) ----
mids = feat_df['molecule_id'].unique()
train_mids, eval_mids = train_test_split(mids, test_size=0.2, random_state=42)
train_df = feat_df[feat_df.molecule_id.isin(train_mids)].reset_index(drop=True)
eval_df  = feat_df[feat_df.molecule_id.isin(eval_mids)].reset_index(drop=True)
print(f'train: {len(train_df):,} rows / {len(train_mids):,} molecules')
print(f'eval:  {len(eval_df):,} rows / {len(eval_mids):,} molecules')

# ---- features ----
# FIX: no 'c1_is_true'. c1_sim is kept (constant per molecule in 19a; 19c will match).
FEATS = ['t_norm', 'a_norm', 'mass_ppm', 'n_cand', 'c1_sim']
print(f'features: {FEATS}')

X_tr = train_df[FEATS].values.astype(np.float32)
y_tr = train_df['label'].values.astype(np.int32)
X_ev = eval_df[FEATS].values.astype(np.float32)
y_ev = eval_df['label'].values.astype(np.int32)

# ---- train ----
clf = HistGradientBoostingClassifier(
    max_depth=6,
    max_iter=500,
    learning_rate=0.05,
    min_samples_leaf=80,
    l2_regularization=1.0,
    random_state=42,
)
clf.fit(X_tr, y_tr)
print('ranker trained')

# ---- evaluate MRR@25 ----
eval_df = eval_df.copy()
eval_df['p'] = clf.predict_proba(X_ev)[:, 1]

mrr_ranker = []
mrr_transformer = []
mrr_c1 = []

for mid, g in eval_df.groupby('molecule_id'):
    g_r = g.sort_values('p', ascending=False).reset_index(drop=True)
    r = g_r.index[g_r.label == 1]
    mrr_ranker.append(1.0 / (r[0] + 1) if len(r) else 0.0)
    g_t = g.sort_values('t_norm', ascending=False).reset_index(drop=True)
    r = g_t.index[g_t.label == 1]
    mrr_transformer.append(1.0 / (r[0] + 1) if len(r) else 0.0)
    mrr_c1.append(1.0 if (g['c1_is_true'] == 1).any() else 0.0)

print(f'\neval MRR@25 (v2, no c1_is_true):')
print(f'  transformer-only : {np.mean(mrr_transformer):.4f}')
print(f'  Channel 1 flag   : {np.mean(mrr_c1):.4f}  (diagnostic only, not a feature)')
print(f'  ranker           : {np.mean(mrr_ranker):.4f}')

with open('/kaggle/working/ranker.pkl', 'wb') as f:
    pickle.dump({'clf': clf, 'features': FEATS}, f)
print('written ranker.pkl')

raw rows: 213,709  molecules: 1,990
after filter: rows 211,824  molecules 1,944
train: 169,940 rows / 1,555 molecules
eval:  41,884 rows / 389 molecules
features: ['t_norm', 'a_norm', 'mass_ppm', 'n_cand', 'c1_sim']
ranker trained

eval MRR@25 (v2, no c1_is_true):
  transformer-only : 0.5706
  Channel 1 flag   : 1.0000  (diagnostic only, not a feature)
  ranker           : 0.9108
written ranker.pkl


In [61]:
# Cell 19c — inference with LightGBM ranker (v2)
# FIX: c1_sim now constant per molecule (matches 19a training).
#      c1_is_true removed entirely (not a trainable feature at test time).
# Channel 1 exact-match override on top.

import pickle
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm
from rdkit.Chem.Descriptors import ExactMolWt
from rdkit import Chem as _Chem

with open('/kaggle/working/ranker.pkl', 'rb') as f:
    _rk = pickle.load(f)
clf = _rk['clf']
FEATS = _rk['features']
print(f'ranker loaded, features: {FEATS}')

# FIX: ensure test has neutral_mass and mols comes from test (not 19a holdout)
test['neutral_mass'] = neutral_mass(
    test['precursor_mz'].values.astype(np.float64),
    test['adduct'].values
)
mols = list(test.groupby('molecule_id', sort=False))
assert len(mols) == 400, f'expected 400 test molecules, got {len(mols)}'

submission_rows = []
n_c1_promoted = 0
n_c1_ranked1 = 0

for gi, (mid, sub) in enumerate(tqdm(mols, desc='ranker inference')):
    nms = sub['neutral_mass'].values[np.isfinite(sub['neutral_mass'].values)]
    if len(nms) == 0:
        submission_rows.append({'molecule_id': mid, 'smiles': 'CCO'})
        continue
    target = float(np.median(nms))
    spec_iter = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]
    c1_ik, c1_sim = channel1_best_ik(target, spec_iter)

    # ---- transformer scores ----
    t_scores = transformer_scores.get(mid, {})

    # ---- analog scores ----
    a_hits = analog_scores(target, spec_iter, topn=100)
    a_scores = dict(a_hits) if a_hits else {}

    # ---- candidate window ----
    lo, hi = mass_window(target, ppm=10.0)
    if hi <= lo:
        lo, hi = mass_window(target, ppm=35.0)
    if hi <= lo:
        submission_rows.append({'molecule_id': mid, 'smiles': 'CCO'})
        continue
    cand_iks = pool_ik_sorted[lo:hi]
    cand_smiles = pool_smiles_sorted[lo:hi]

    # dedup (ik, smiles)
    _seen = set()
    u_ik, u_smi = [], []
    for ik, smi in zip(cand_iks, cand_smiles):
        if ik not in _seen:
            _seen.add(ik)
            u_ik.append(ik)
            u_smi.append(smi)
    cand_iks, cand_smiles = u_ik, u_smi
    if not cand_iks:
        submission_rows.append({'molecule_id': mid, 'smiles': 'CCO'})
        continue

    t_max = max(t_scores.values()) if t_scores else 1.0
    a_max = max(a_scores.values()) if a_scores else 1.0

    # FIX: c1_sim is constant per molecule (matches 19a), not a delta on c1 candidate.
    c1_sim_val = float(c1_sim) if c1_ik is not None else 0.0
    c1_idx = None

    # ---- build feature matrix ----
    X = np.zeros((len(cand_iks), len(FEATS)), dtype=np.float32)
    for j, (ik, smi) in enumerate(zip(cand_iks, cand_smiles)):
        if c1_ik is not None and ik == c1_ik:
            c1_idx = j
        m = _Chem.MolFromSmiles(smi)
        ppm_err = abs(ExactMolWt(m) - target) / target * 1e6 if m is not None else 1e6
        for k, f in enumerate(FEATS):
            if f == 't_norm':     X[j, k] = t_scores.get(ik, 0.0) / (t_max + 1e-9)
            elif f == 'a_norm':   X[j, k] = a_scores.get(ik, 0.0) / (a_max + 1e-9)
            elif f == 'mass_ppm': X[j, k] = ppm_err
            elif f == 'n_cand':   X[j, k] = len(cand_iks)
            elif f == 'c1_sim':   X[j, k] = c1_sim_val   # constant per molecule

    p = clf.predict_proba(X)[:, 1]
    order = np.argsort(-p)[:25]
    smiles_25 = [cand_smiles[i] for i in order]

    # diagnostic: did ranker put c1 at top-1 before override?
    if c1_idx is not None and order[0] == c1_idx:
        n_c1_ranked1 += 1

    # ---- Channel 1 override (identical to 18a) ----
    if c1_ik is not None and c1_sim > TAU:
        c1_smiles = ik_to_smiles.get(c1_ik)
        if c1_smiles is not None:
            smiles_25 = [c1_smiles] + [s for s in smiles_25 if s != c1_smiles]
            smiles_25 = smiles_25[:25]
            n_c1_promoted += 1

    while len(smiles_25) < 25:
        smiles_25.append(smiles_25[-1] if smiles_25 else 'CCO')

    submission_rows.append({'molecule_id': mid, 'smiles': ';'.join(smiles_25)})

sub = pd.DataFrame(submission_rows)
print(f'\nChannel 1 promoted: {n_c1_promoted}/{len(mols)}')
print(f'Channel 1 already ranker-top1 before override: {n_c1_ranked1}/{len(mols)}')
print(sub.shape)
print(sub.head())

assert set(sub['molecule_id']) == set(test['molecule_id']), 'molecule_id mismatch'
max_g = sub['smiles'].apply(lambda s: len(s.split(';'))).max()
assert max_g <= 25, f'too many guesses: {max_g}'
sub.to_csv('submission.csv', index=False)
print('written submission.csv')

ranker loaded, features: ['t_norm', 'a_norm', 'mass_ppm', 'n_cand', 'c1_sim']


ranker inference:   0%|          | 0/400 [00:00<?, ?it/s]


Channel 1 promoted: 400/400
Channel 1 already ranker-top1 before override: 329/400
(400, 2)
  molecule_id                                             smiles
0    m_005e53  CN(Cc1ncccc1C(=O)O)CC1(c2ccc(Br)cc2)CC1;CC1CC(...
1    m_006153  COc1ccc(N2CC(C(=O)Nc3nncn3C3CC3)CC2=O)cc1;COc1...
2    m_00b5aa  COc1ccnc(CN2CCOCC3(CCCC3)C2)c1OC;CC(C)(C)C(=O)...
3    m_01e922  Cc1ccc(C(=O)OCC2CCCCC2)c(=O)[nH]1;COC[C@@H]1CC...
4    m_0259d4  O=C(NCCCn1ccnc1)NCCc1coc(-c2ccccc2)n1;Cc1ccc(-...
written submission.csv


In [56]:
# Cell 20

print(ik_to_smiles.get('QCTQNXJUUXYSRK'))
print(ik_to_smiles.get('BKIHGKNEJKYYAC'))
print(ik_to_smiles.get('RWCPIENWOXGKAV'))

CN(Cc1ncccc1C(=O)O)CC1(c2ccc(Br)cc2)CC1
COc1ccc(N2CC(C(=O)Nc3nncn3C3CC3)CC2=O)cc1
COc1ccnc(CN2CCOCC3(CCCC3)C2)c1OC


In [58]:
import os; print(os.listdir('/kaggle/working'))

import pandas as pd
df = pd.read_parquet('/kaggle/working/pool_b_features.parquet')
pos_per_mol = df.groupby('molecule_id')['label'].sum()
print(pos_per_mol.value_counts().sort_index())

['.virtual_documents', 'submission.csv', 'ranker.pkl', 'pool_b_features.parquet']
label
0      46
1    1944
Name: count, dtype: int64
